## Getting time between events (regardless of tauc exceedance)

In [1]:
import pandas as pd

# import event tables
spring_events = pd.read_csv("spring_events/spring_events.csv", parse_dates=["start", "end", "peak_time"])
summer_events = pd.read_csv("summer_events/summer_events.csv", parse_dates=["start", "end", "peak_time"])
# combine seasons
all_events = pd.concat([spring_events, summer_events], ignore_index=True)
# sort chronologically
all_events = all_events.sort_values("peak_time").reset_index(drop=True)

In [2]:
# calculate time since previous peak
all_events["time_since_previous"] = (all_events["peak_time"].diff())
# convert to hours
all_events["hours_since_previous"] = (all_events["time_since_previous"].dt.total_seconds() / 3600)
# convert to days
all_events["days_since_previous"] = (all_events["time_since_previous"].dt.total_seconds() / 86400)
print(all_events[
    [
        "storm",
        "peak_time",
        "time_since_previous",
        "hours_since_previous",
        "days_since_previous"
    ]
])

   storm           peak_time time_since_previous  hours_since_previous  \
0    st1 2021-07-23 16:30:00                 NaT                   NaN   
1    st2 2022-08-03 16:15:00   375 days 23:45:00               9023.75   
2    st3 2022-08-08 14:45:00     4 days 22:30:00                118.50   
3    NaN 2023-04-17 22:18:00   252 days 07:33:00               6055.55   
4    NaN 2023-04-18 20:18:00     0 days 22:00:00                 22.00   
5    NaN 2023-04-19 18:03:00     0 days 21:45:00                 21.75   
6    NaN 2023-04-21 18:33:00     2 days 00:30:00                 48.50   
7    NaN 2023-04-22 23:18:00     1 days 04:45:00                 28.75   
8    NaN 2023-04-23 20:18:00     0 days 21:00:00                 21.00   
9    NaN 2023-04-24 17:48:00     0 days 21:30:00                 21.50   
10   NaN 2023-04-27 20:48:00     3 days 03:00:00                 75.00   
11   NaN 2023-04-29 20:18:00     1 days 23:30:00                 47.50   
12   NaN 2023-04-30 19:48:00     0 day

In [3]:
all_events.to_csv("time_between_events.csv", index=False)

## Getting time between events with tauc exceedance

In [4]:
# import event-tauc summary tables
spring_events = pd.read_csv(
    "spring_events/spring_event_tauc_summary.csv", parse_dates=["event_start", "event_peak", "event_end"])
summer_events = pd.read_csv("summer_events/summer_event_tauc_summary.csv", parse_dates=["storm_start", "storm_peak", "storm_end"])

# rename columns so spring and summer match
spring_events = spring_events.rename(columns={
    "event": "event_name",
    "event_start": "start",
    "event_peak": "peak_time",
    "event_end": "end"
})
summer_events = summer_events.rename(columns={
    "storm": "event_name",
    "storm_start": "start",
    "storm_peak": "peak_time",
    "storm_end": "end"
})
# combine seasons
all_events = pd.concat([spring_events, summer_events], ignore_index=True)
# keep only events/storms with tauc exceedance
tauc_events = all_events[all_events["has_tauc_exceedance"] == True].copy()
# sort chronologically
tauc_events = tauc_events.sort_values("peak_time").reset_index(drop=True)


In [5]:
# calculate time since previous tauc-exceeding event peak
tauc_events["time_since_previous_tauc_peak"] = tauc_events["peak_time"].diff()
# convert to hours
tauc_events["hours_since_previous_tauc_peak"] = (
    tauc_events["time_since_previous_tauc_peak"].dt.total_seconds() / 3600
)
# convert to days
tauc_events["days_since_previous_tauc_peak"] = (
    tauc_events["time_since_previous_tauc_peak"].dt.total_seconds() / 86400
)
print(tauc_events[
    [
        "event_name",
        "peak_time",
        "has_tauc_exceedance",
        "time_since_previous_tauc_peak",
        "hours_since_previous_tauc_peak",
        "days_since_previous_tauc_peak"
    ]
])

   event_name           peak_time  has_tauc_exceedance  \
0         st2 2022-08-03 16:15:00                 True   
1         st3 2022-08-08 14:45:00                 True   
2           3 2023-04-19 18:03:00                 True   
3           4 2023-04-21 18:33:00                 True   
4           5 2023-04-22 23:18:00                 True   
5           6 2023-04-23 20:18:00                 True   
6           8 2023-04-27 20:48:00                 True   
7           9 2023-04-29 20:18:00                 True   
8          10 2023-04-30 19:48:00                 True   
9          12 2023-05-02 19:33:00                 True   
10         13 2023-05-03 19:18:00                 True   
11         14 2023-05-04 18:48:00                 True   
12        st4 2023-07-29 15:45:00                 True   
13        st7 2023-09-14 18:00:00                 True   

   time_since_previous_tauc_peak  hours_since_previous_tauc_peak  \
0                            NaT                             

In [6]:
tauc_events.to_csv("time_between_tauc_exceedance_events.csv", index=False)
